In [1]:
import re
import pandas as pd
import numpy as np
import nltk
import pymorphy3
import optuna
import mlflow
import pickle
import implicit

from implicit.als import AlternatingLeastSquares
from scipy.sparse import coo_matrix, csr_matrix, vstack
from optuna.integration.mlflow import MLflowCallback
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ndcg_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from nltk.corpus import stopwords
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from gensim.utils import simple_preprocess
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.simplefilter('ignore', FutureWarning)

from sklearn import set_config
set_config(display='diagram')

In [2]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = "hr-ai-scout"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

# Load data

In [4]:
df = pd.read_csv('/Users/user/Documents/Magistracy/year_project/hr-ai-scout/total_df.csv')
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_salary_from,vacancy_salary_to,vacancy_salary_currency,vacancy_salary_gross,...,resume_education,resume_courses,resume_salary,resume_age,resume_total_experience,resume_experience_months,resume_location,resume_gender,resume_applicant_status,target
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Казанский Авиационный Институт'],NaN,NaN,65.0,19 лет,228.0,Москва,Мужчина,Рассматривает предложения,1
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,"['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...","['ООО ""Открытый Учебный Центр СофтБаланс"", г. ...",NaN,43.0,17 лет 4 месяца,208.0,Москва,Мужчина,Рассматривает предложения,1
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Орский государственный педагогический инстит...,NaN,200 000 ₽ на руки,52.0,30 лет,360.0,Москва,Женщина,NaN,1
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Красноярский государственный университет'],NaN,500 000 ₽ на руки,56.0,29 лет 8 месяцев,356.0,Красноярск,Мужчина,Рассматривает предложения,1
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,300000.0,NaN,RUR,False,...,['Белоруский Гос. Университет Информатики и Ра...,"['SAP CIS, SAP XI', 'Школа Логистики МАДИ', 'S...",NaN,48.0,25 лет 1 месяц,301.0,Moscow,Male,NaN,1


# Preprocessing

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
В первую очередь уберем строки, где пропущены все ключевые значения в резюме:
</div>

In [5]:
t1 = df.shape[0]
df = df.dropna(subset= ["resume_education",
                        "resume_last_experience_description",
                        "resume_last_position",
                        "resume_last_company_experience_period",
                        "resume_total_experience",
                        "resume_experience_months",
                        "resume_location",
                        "resume_specialization",
                        # "resume_gender",
                        # "resume_title"
                       ], how="all")
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строки')

Удалено  84  строки


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Удалим еще те строки, где случился технический сбой в парсинге, где у кандидата общий опыт есть, а последний опыт не указан (и наоборот):
</div>

In [6]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].notna()
        & df["resume_last_experience_description"].isna()
        & df["resume_last_position"].isna())]
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строк')

Удалено  1543  строк


In [7]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].isna()
        & df["resume_last_experience_description"].notna()
        & df["resume_last_position"].notna())]
t2 = df.shape[0]
print('Удалено ', t1 - t2 ,' строк')

Удалено  0  строк


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Посмотрим на пропуски отдельно по категориальным и числовым признакам.
</div>

In [8]:
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns

In [9]:
df[cat_cols] = df[cat_cols].fillna('NDT')

In [10]:
df.loc[df['resume_experience_months'].isna(), 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

In [11]:
df['resume_age'] = df['resume_age'].fillna(df['resume_age'].mean())
df['resume_experience_months'] = df['resume_experience_months'].fillna(0)

In [12]:
df = df.drop(['vacancy_salary_to', 'vacancy_salary_from',
              'vacancy_salary_currency', 'vacancy_salary_gross'], axis=1)

In [13]:
df.loc[df['resume_last_company_experience_period'] == 'NDT', 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Преобразуем сначала ожидаемые зарплаты
</div>

In [14]:
df['resume_salary_split'] = df['resume_salary'].apply(lambda x: x.split())

df['salary_int'] = df['resume_salary_split'].apply(
    lambda x: int(''.join(part for part in x if re.fullmatch(r'\d+', part)))
              if any(re.fullmatch(r'\d+', part) for part in x)
              else np.nan
)

currency_symbols = ['₽', '$', '€', '₴', '₸', '₼', '₾', 'Br', "so'm"]

rates_rub = {
    "₽": 1.0,
    "$": 80.85,
    "€": 94.14,
    "₴": 1.94,
    "₸": 0.150,
    "₼": 47.8,
    "₾": 33.5,
    "Br": 28.7,
    "so'm": 0.0068
}

df['currency_symbol'] = df['resume_salary_split'].apply(
    lambda x: next((sym for sym in x if sym in currency_symbols), np.nan)
)

df['salary_converted'] = (df['salary_int'] * df['currency_symbol'].map(rates_rub)).fillna(0)

df['resume_salary'] = df['salary_converted']

df = df.drop(['resume_salary_split', 'salary_int', 'currency_symbol', 'salary_converted'], axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Добавим дополнительный столбец с опытом работы в последней компании в месяцах для удобства
</div>

In [15]:
def experience_to_months(experience_text):
    months = 0
    # Опыт в годах
    years_match = re.search(r'(\d+)\s*год', experience_text)
    if years_match:
        months += int(years_match.group(1)) * 12

    years_match = re.search(r'(\d+)\s*лет', experience_text)
    if years_match:
        months += int(years_match.group(1)) * 12

    # Опыт в месяцах
    months_match = re.search(r'(\d+)\s*месяц', experience_text)
    if months_match:
        months += int(months_match.group(1))

    return months if months > 0 else np.nan

In [16]:
df['resume_last_company_experience_months'] = df['resume_last_company_experience_period'].apply(experience_to_months)

In [17]:
df.loc[df['resume_last_company_experience_period'] == 'NDT', 'resume_last_experience_description'].unique()

array(['NDT'], dtype=object)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Т.к. в названии компании стоит NDT, можно столбец resume_last_company_experience_months заполнять нулевыми значениями.
</div>

In [18]:
df['resume_last_company_experience_months'] = df['resume_last_company_experience_months'].fillna(0)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

- Ограничим выбросы по зарплате, потому что ровно одно значение по ожидаемой заработоной плате = 999,999,999 (смешно, но нет)

- Ограничим опыт общий и внутри одной компании до 720 месяцев (60 лет, ничего себе уже)

- Уберем возраст > 90, не ждем, что эти кандидаты находятся в поиске вакансии
</div>

In [19]:
df = df[~(df.resume_salary > 1e7)]
df.loc[df['resume_experience_months'] > 720, 'resume_experience_months'] = 720
df.loc[df['resume_last_company_experience_months'] > 720, 'resume_last_company_experience_months'] = 720
df = df[~(df.resume_age > 90)]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

- Также уберем строки, где последний опыт кандидата больше, чем общий

- И где общий опыт кандидата +16 лет больше чем возраст (хоть так)

</div>

In [20]:
df = df[~(df.resume_experience_months < df.resume_last_company_experience_months)]
df = df[~(df.resume_age < (df.resume_experience_months // 12) + 16)]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Заменим текущий формат разброса полов в датасете на унифицированный

</div>

In [21]:
gender_map = {
    'Мужчина': 'Мужчина',
    'Male': 'Мужчина',
    'Женщина': 'Женщина',
    'Female': 'Женщина'
}

df['resume_gender'] = df['resume_gender'].apply(lambda x: gender_map[x] if x in gender_map else 'Неизвестно')

In [22]:
print(f"Датасет после очистки: {df.shape}")

Датасет после очистки: (325543, 25)


# Feature engineering

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">
Добавим признак матчинга локации вакансии и резюме

</div>

In [23]:
df['location_matching'] = df.apply(lambda row: 1 if row['vacancy_area'] == row['resume_location'] else 0, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Сделаем новый признак, а именно посчитаем количество навыков кандидата, которые указаны в вакансии.

</div>

In [24]:
def resume_skill_count_in_vacancy(row):
    count = 0
    skill_list = row['resume_skills'].replace('[', '').replace(']', '').replace("'", "").split(', ')
    for i in skill_list:
        if i in row['vacancy_description']:
            count += 1
    return count

df['resume_skill_count_in_vacancy'] = df.apply(resume_skill_count_in_vacancy, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Также посчитаем долю слов из последней должности в резюме, которые указаны в вакансии.

</div>

In [25]:
def last_position_in_vacancy(row):
    bow = []
    seps = [' ', '-', '_']
    for sep in seps:
        bow += row['resume_last_position'].split(sep=sep)
        bow = list(set(bow))
    
    c = 0
    for word in bow:
        if word in row['vacancy_description']:
            c +=1
    
    return c / len(bow)

In [26]:
df['last_position_in_vacancy'] = df.apply(last_position_in_vacancy, axis=1)

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Теперь закодируем описание вакансии и последнего опыта работы и сравним через косинусное расстояние.

</div>

In [27]:
def preprocess_data(df):
    """Обработка пропущенных значений в текстовых полях"""
    print("Проверка пропущенных значений...")
    print(f"Пропуски в vacancy_description: {df['vacancy_description'].isna().sum()}")
    print(f"Пропуски в resume_last_experience_description: {df['resume_last_experience_description'].isna().sum()}")
    
    # Заполняем пропуски пустыми строками
    df['vacancy_description'] = df['vacancy_description'].fillna('')
    df['resume_last_experience_description'] = df['resume_last_experience_description'].fillna('')
    
    # Проверяем, что все значения теперь строковые
    df['vacancy_description'] = df['vacancy_description'].astype(str)
    df['resume_last_experience_description'] = df['resume_last_experience_description'].astype(str)
    
    return df

In [28]:
def save_results(df, output_file):
    """Сохранение результатов в CSV файл"""
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Результаты сохранены в файл: {output_file}")

In [29]:
def calculate_cosine_similarity(embeddings1, embeddings2):
    """Вычисление косинусного сходства между двумя наборами эмбеддингов"""
    similarities = []
    
    for i in tqdm(range(embeddings1.shape[0])):
        emb1_row = embeddings1[i]
        emb2_row = embeddings2[i]
        
        similarity = cosine_similarity(emb1_row, emb2_row)[0][0]
        similarities.append(similarity)
    
    return similarities

In [30]:
warnings.filterwarnings('ignore')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('taggers/averaged_perceptron_tagger_ru')
except LookupError:
    nltk.download('averaged_perceptron_tagger_ru')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

morph = pymorphy3.MorphAnalyzer()

[nltk_data] Downloading package wordnet to /Users/user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [31]:
def lemmatize_russian(tokens):
    """Лемматизация русских слов"""
    lemmas = []
    for token in tokens:
        parsed = morph.parse(token)[0]  # Берем самый вероятный разбор
        lemmas.append(parsed.normal_form)
    return lemmas

In [32]:
def tokenize_and_lemmatize(text):
    """Токенизация текста с лемматизацией и удалением стоп-слов"""
    tokens = simple_preprocess(text, deacc=True, min_len=2)
    stop_words = set(stopwords.words('russian') + stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    lemmatized_tokens = lemmatize_russian(tokens)
    
    return lemmatized_tokens

In [33]:
def get_tfidf_embeddings(texts, vectorizer=None, fit=True):
    """Создание TF-IDF эмбеддингов для списка текстов с лемматизацией"""
    if fit:
        vectorizer = TfidfVectorizer(
            max_features=5000,
            min_df=2,
            max_df=0.8,
            ngram_range=(1, 2),
            tokenizer=tokenize_and_lemmatize,
            token_pattern=None,
            lowercase=False  # Уже сделано в токенизации
        )
        embeddings = vectorizer.fit_transform(texts)
    else:
        embeddings = vectorizer.transform(texts)
    
    return embeddings, vectorizer

In [34]:
def get_tfidf_vacancy_embeddings(df, vectorizer=None):
    """Создание эмбеддингов для уникальных вакансий с лемматизацией"""
    unique_vacancies = df[['vacancy_id', 'vacancy_description']].drop_duplicates()
    
    unique_embeddings, vectorizer = get_tfidf_embeddings(
        unique_vacancies['vacancy_description'].tolist(), 
        vectorizer=vectorizer, 
        fit=(vectorizer is None)
    )
    
    vacancy_embedding_dict = dict(zip(unique_vacancies['vacancy_id'], unique_embeddings))
    
    rows = []
    for vid in df['vacancy_id']:
        rows.append(vacancy_embedding_dict[vid])
    
    all_vacancy_embeddings = vstack(rows)
    
    return all_vacancy_embeddings, vectorizer

In [35]:
def process_similarity_scores_tfidf(df, vectorizer=None, fit=True):
    """Функция для вычисления схожести с использованием TF-IDF и лемматизации"""
    df = preprocess_data(df)
    
    print("Создание TF-IDF эмбеддингов для описаний опыта в резюме...")
    experience_embeddings, tfidf_vectorizer = get_tfidf_embeddings(df['resume_last_experience_description'].tolist(), vectorizer=vectorizer, fit=fit)
    
    print("Создание TF-IDF эмбеддингов для описаний вакансий...")
    vacancy_embeddings, _ = get_tfidf_vacancy_embeddings(df, vectorizer=tfidf_vectorizer)
    
    print("Вычисление косинусного сходства...")
    similarity_scores = calculate_cosine_similarity(vacancy_embeddings, experience_embeddings)
    
    df['similarity_score_tfidf'] = similarity_scores
    
    return df, tfidf_vectorizer

In [36]:
try:
    df_tfidf = pd.read_csv('description_df_with_scores_tfidf.csv')
except:
    df_tfidf = process_similarity_scores_tfidf(df.copy())
    save_results(df_tfidf, 'description_df_with_scores_tfidf.csv')

In [37]:
df = df.merge(df_tfidf)

In [38]:
df.head()

,vacancy_id,vacancy_name,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,vacancy_description,resume_id,resume_title,resume_specialization,...,resume_location,resume_gender,resume_applicant_status,target,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,resume_skill_count,similarity_score_tfidf
0,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",6969174,ABAP-разработчик,"['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,76.0,1,3,0.666667,3,0.284047
1,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",9100077,"ABAP разработчик - SAP HCM, CRM, S/4HANA ERP(F...","['Программист, разработчик']",...,Москва,Мужчина,Рассматривает предложения,1,8.0,1,2,0.500000,2,0.308726
2,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",32644957,Разработчик ABAP,"['Программист, разработчик']",...,Москва,Женщина,NDT,1,136.0,1,1,0.000000,1,0.510093
3,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",27220466,ABAP-разработчик,"['Программист, разработчик']",...,Красноярск,Мужчина,Рассматривает предложения,1,135.0,0,2,0.333333,2,0.301062
4,126167948,Разработчик SAP ABAP,Москва,Более 6 лет,Полная занятость,Удаленная работа,"Привет!.redev — технологическая компания, созд...",7532708,ABAP разработчик. Senior ABAP Developer. SAP T...,"['Programmer, developer']",...,Moscow,Мужчина,NDT,1,0.0,0,2,0.600000,2,0.075429


# Train-test split

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Добавим новые признаки в обучение и тестирование

</div>

In [39]:
features = [
    'vacancy_area',
    'vacancy_experience',
    'vacancy_employment', 
    'vacancy_schedule',
    # 'resume_specialization',
    # 'resume_education', 
    # 'resume_courses', 
    'resume_salary',
    'resume_age', 
    'resume_experience_months',
    'resume_location',
    'resume_gender', 
    'resume_applicant_status', 
    'resume_last_company_experience_months', 
    'location_matching',
    'resume_skill_count_in_vacancy',
    'last_position_in_vacancy',
    'similarity_score_tfidf'
]
df[features]

,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,resume_salary,resume_age,resume_experience_months,resume_location,resume_gender,resume_applicant_status,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,similarity_score_tfidf
0,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,65.000000,228.0,Москва,Мужчина,Рассматривает предложения,76.0,1,3,0.666667,0.284047
1,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,43.000000,208.0,Москва,Мужчина,Рассматривает предложения,8.0,1,2,0.500000,0.308726
2,Москва,Более 6 лет,Полная занятость,Удаленная работа,200000.0,52.000000,360.0,Москва,Женщина,NDT,136.0,1,1,0.000000,0.510093
3,Москва,Более 6 лет,Полная занятость,Удаленная работа,500000.0,56.000000,356.0,Красноярск,Мужчина,Рассматривает предложения,135.0,0,2,0.333333,0.301062
4,Москва,Более 6 лет,Полная занятость,Удаленная работа,0.0,48.000000,301.0,Moscow,Мужчина,NDT,0.0,0,2,0.600000,0.075429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321637,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,242550.0,66.000000,521.0,Санкт-Петербург,Женщина,NDT,270.0,0,0,0.166667,0.072670
321638,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,40.000000,213.0,Москва,Мужчина,Активно ищет работу,35.0,1,0,0.000000,0.000000
321639,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,80000.0,44.060813,121.0,Москва,Мужчина,NDT,44.0,1,0,0.200000,0.047398
321640,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,32.000000,117.0,Москва,Женщина,NDT,96.0,1,0,0.200000,0.029086


In [40]:
numeric_features = df[features].select_dtypes(include=np.number).columns
categorical_features = df[features].select_dtypes(exclude=np.number).columns

In [41]:
X = df[features].copy()
y = df['target'].copy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [42]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((257313, 15), (64329, 15), (257313,), (64329,))

In [43]:
def calculate_metrics(df_test: pd.DataFrame) -> pd.DataFrame:
    ndcg_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    vacancy_ids = df_test['vacancy_id'].unique()
    
    for vacancy_id in vacancy_ids:
        mask = df_test['vacancy_id'] == vacancy_id
        y_true = df_test.loc[mask, 'target'].values
        y_score = df_test.loc[mask, 'y_pred_proba'].values
        
        if len(y_true) <= 1:
            continue
        
        y_true_2d = y_true.reshape(1, -1)
        y_score_2d = y_score.reshape(1, -1)
        
        ndcg = ndcg_score(y_true_2d, y_score_2d)
        ndcg_scores.append(ndcg)
        
        y_pred_binary = (y_score >= 0.5).astype(int)
        
        precision = precision_score(y_true, y_pred_binary, zero_division=0)
        recall = recall_score(y_true, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true, y_pred_binary, zero_division=0)
        
        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
    
    if ndcg_scores:
        print(f"Средний NDCG: {np.mean(ndcg_scores):.4f}")
        print(f"Средний Precision: {np.mean(precision_scores):.4f}")
        print(f"Средний Recall: {np.mean(recall_scores):.4f}")
        print(f"Средний F1-Score: {np.mean(f1_scores):.4f}")

        return np.mean(ndcg_scores), np.mean(precision_scores), np.mean(recall_scores), np.mean(f1_scores)
    else:
        print("Недостаточно данных для расчета метрик")

        return None, None, None, None

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Обучим с подбором гиперпараметров `LogisticRegression`, как бейзлайн для сравнения с нелинейными моделями, `RandomForestClassifier`, как разновидность бэггинга, и три разные вида градиентного бустинга: `XGBClassifier`, `LGBMClassifier` и `CatBoostClassifier`.

</div>

# LogisticRegression

In [44]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'model__C': trial.suggest_float('C', 1e-4, 1e4, log=True),
        'model__penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
        'model__solver': trial.suggest_categorical('solver', ['liblinear', 'saga'])
    }
    
    pipeline_lr_optuna = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
        ])),
        ('model', LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    
    pipeline_lr_optuna.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lr_optuna.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lr_optuna.predict_proba(X_fold_val)
        

        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)

        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [45]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE = 'LogisticRegression_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

🏃 View run LogisticRegression_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/ae4318e68be745cab3d061837f4e3e02
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [46]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "LogisticRegression_optuna"

mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id}}
)

In [47]:
study = optuna.create_study(direction='maximize', 
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            study_name=STUDY_NAME,
                            storage=STUDY_DB_NAME,
                            load_if_exists=True)
study.optimize(objective, 
               n_trials=10, 
               callbacks=[mlflc]
              )
best_params_optuna = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params_optuna}")

[I 2026-05-04 00:36:28,558] Using an existing study with name 'LogisticRegression_optuna' instead of creating a new one.


Средний NDCG: 0.8302
Средний Precision: 0.5690
Средний Recall: 0.7892
Средний F1-Score: 0.6305
Средний NDCG: 0.8191
Средний Precision: 0.5621
Средний Recall: 0.7796
Средний F1-Score: 0.6219


[I 2026-05-04 00:37:16,836] Trial 61 finished with value: 0.8270530007353442 and parameters: {'C': 0.4956726270237583, 'penalty': 'l1', 'solver': 'saga'}. Best is trial 30 with value: 0.8272130448068064.


Средний NDCG: 0.8271
Средний Precision: 0.5719
Средний Recall: 0.7820
Средний F1-Score: 0.6308
🏃 View run 61 at: http://127.0.0.1:5000/#/experiments/1/runs/c4c287a899e14f80a6b31034654ea72b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8302
Средний Precision: 0.5693
Средний Recall: 0.7897
Средний F1-Score: 0.6309
Средний NDCG: 0.8192
Средний Precision: 0.5623
Средний Recall: 0.7797
Средний F1-Score: 0.6222


[I 2026-05-04 00:38:16,199] Trial 62 finished with value: 0.8270976678687342 and parameters: {'C': 0.9248049576299547, 'penalty': 'l1', 'solver': 'saga'}. Best is trial 30 with value: 0.8272130448068064.


...

Средний NDCG: 0.8271
Средний Precision: 0.5714
Средний Recall: 0.7820
Средний F1-Score: 0.6305
🏃 View run 68 at: http://127.0.0.1:5000/#/experiments/1/runs/3cba0ba492bd4e9aae6c4d265b3fc579
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8302
Средний Precision: 0.5685
Средний Recall: 0.7889
Средний F1-Score: 0.6302
Средний NDCG: 0.8189
Средний Precision: 0.5620
Средний Recall: 0.7799
Средний F1-Score: 0.6223


[I 2026-05-04 00:46:23,320] Trial 69 finished with value: 0.826703500703733 and parameters: {'C': 2.9745727618923943, 'penalty': 'l1', 'solver': 'liblinear'}. Best is trial 30 with value: 0.8272130448068064.


Средний NDCG: 0.8267
Средний Precision: 0.5707
Средний Recall: 0.7818
Средний F1-Score: 0.6298
🏃 View run 69 at: http://127.0.0.1:5000/#/experiments/1/runs/3dccd2310cd440a4974b6a72ef60c388
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8302
Средний Precision: 0.5691
Средний Recall: 0.7890
Средний F1-Score: 0.6305
Средний NDCG: 0.8193
Средний Precision: 0.5627
Средний Recall: 0.7798
Средний F1-Score: 0.6222


[I 2026-05-04 00:46:58,884] Trial 70 finished with value: 0.8269231121485823 and parameters: {'C': 0.18384129366558707, 'penalty': 'l1', 'solver': 'saga'}. Best is trial 30 with value: 0.8272130448068064.


Средний NDCG: 0.8269
Средний Precision: 0.5708
Средний Recall: 0.7819
Средний F1-Score: 0.6300
🏃 View run 70 at: http://127.0.0.1:5000/#/experiments/1/runs/36cfaf56223c49ac8aebf5eae58f0ee3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 71
Best params: {'C': 0.18851801168357057, 'penalty': 'l1', 'solver': 'liblinear'}


In [48]:
study.best_trial.user_attrs['pipeline_params']

{'model__C': 0.18851801168357057,
 'model__penalty': 'l1',
 'model__solver': 'liblinear'}

In [49]:
pipeline_lr_best_optuna = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('numeric_scaling', StandardScaler(), numeric_features),
        ('categorical_encoding', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])),
    ('model', LogisticRegression(random_state=RANDOM_STATE))
])

pipeline_lr_best_optuna.set_params(**study.best_trial.user_attrs['pipeline_params'])
pipeline_lr_best_optuna.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_scaling', ...), ('categorical_encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [50]:
y_pred_proba = pipeline_lr_best_optuna.predict_proba(X_test)

df_test = df.loc[X_test.index].copy()
df_test['y_pred_proba'] = y_pred_proba[:, 1]

In [51]:
ndcg_lr, precision_lr, recall_lr, f1_lr = calculate_metrics(df_test)

metrics_lr = {
    'NDCG': ndcg_lr,
    'Precision': precision_lr,
    'Recall': recall_lr,
    'F1': f1_lr
}

Средний NDCG: 0.7515
Средний Precision: 0.6386
Средний Recall: 0.5985
Средний F1-Score: 0.6021


In [52]:
RUN_NAME = "best_optuna_lr" 
REGISTRY_MODEL_NAME = "best_optuna_lr"

In [53]:
signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    lr_info = mlflow.sklearn.log_model(sk_model=pipeline_lr_best_optuna, 
                                       artifact_path='best_optuna_lr',
                                       registered_model_name=REGISTRY_MODEL_NAME,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_lr)
    mlflow.log_params(best_params_optuna)

Registered model 'best_optuna_lr' already exists. Creating a new version of this model...
2026/05/04 00:47:13 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lr, version 7


🏃 View run best_optuna_lr at: http://127.0.0.1:5000/#/experiments/1/runs/2fbd6500acbf46ac9bb2dc1e794b5d5b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '7' of model 'best_optuna_lr'.


# RandomForestClassifier

In [54]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 30),
        'model__min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'model__min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20)
    }
    
    pipeline_rf = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ])),
        ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    
    pipeline_rf.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_rf.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_rf.predict_proba(X_fold_val)
        

        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)

        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [55]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE = 'RandomForestClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

🏃 View run RandomForestClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/fdff008978ac4aee8b1e5fffcd8558c9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [56]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME = "RandomForestClassifier_optuna"

mlflc = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id}}
)

In [57]:
study = optuna.create_study(direction='maximize', 
                            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                            study_name=STUDY_NAME,
                            storage=STUDY_DB_NAME,
                            load_if_exists=True)
study.optimize(objective, 
               n_trials=10, 
               callbacks=[mlflc]
              )
best_params_optuna = study.best_params

print(f"Number of finished trials: {len(study.trials)}")
print(f"Best params: {best_params_optuna}")

[I 2026-05-04 00:47:13,363] Using an existing study with name 'RandomForestClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8601
Средний Precision: 0.7506
Средний Recall: 0.8112
Средний F1-Score: 0.7627
Средний NDCG: 0.8476
Средний Precision: 0.7373
Средний Recall: 0.7988
Средний F1-Score: 0.7501


[I 2026-05-04 00:50:47,169] Trial 57 finished with value: 0.8554229332826127 and parameters: {'n_estimators': 950, 'max_depth': 19, 'min_samples_split': 8, 'min_samples_leaf': 2}. Best is trial 57 with value: 0.8554229332826127.


Средний NDCG: 0.8554
Средний Precision: 0.7425
Средний Recall: 0.8024
Средний F1-Score: 0.7547
🏃 View run 57 at: http://127.0.0.1:5000/#/experiments/1/runs/ae98078b3dcd43e78a735bd3a7942190
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8603
Средний Precision: 0.7545
Средний Recall: 0.8110
Средний F1-Score: 0.7650
Средний NDCG: 0.8480
Средний Precision: 0.7421
Средний Recall: 0.7978
Средний F1-Score: 0.7525


[I 2026-05-04 00:54:19,382] Trial 58 finished with value: 0.8557192563717011 and parameters: {'n_estimators': 950, 'max_depth': 21, 'min_samples_split': 8, 'min_samples_leaf': 2}. Best is trial 58 with value: 0.8557192563717011.


...

Средний NDCG: 0.8542
Средний Precision: 0.7628
Средний Recall: 0.7757
Средний F1-Score: 0.7539
🏃 View run 64 at: http://127.0.0.1:5000/#/experiments/1/runs/6240b6830f12417b90faafc862bfc1b9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8603
Средний Precision: 0.7602
Средний Recall: 0.8028
Средний F1-Score: 0.7646
Средний NDCG: 0.8479
Средний Precision: 0.7500
Средний Recall: 0.7904
Средний F1-Score: 0.7541


[I 2026-05-04 01:17:34,176] Trial 65 finished with value: 0.8564977089521156 and parameters: {'n_estimators': 800, 'max_depth': 21, 'min_samples_split': 3, 'min_samples_leaf': 2}. Best is trial 65 with value: 0.8564977089521156.


Средний NDCG: 0.8565
Средний Precision: 0.7567
Средний Recall: 0.7938
Средний F1-Score: 0.7596
🏃 View run 65 at: http://127.0.0.1:5000/#/experiments/1/runs/b56a4f0429ce44aea841e73419bb6059
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8603
Средний Precision: 0.7602
Средний Recall: 0.8028
Средний F1-Score: 0.7646
Средний NDCG: 0.8479
Средний Precision: 0.7500
Средний Recall: 0.7904
Средний F1-Score: 0.7541


[I 2026-05-04 01:51:40,332] Trial 66 finished with value: 0.8564977089521156 and parameters: {'n_estimators': 800, 'max_depth': 21, 'min_samples_split': 3, 'min_samples_leaf': 2}. Best is trial 65 with value: 0.8564977089521156.


Средний NDCG: 0.8565
Средний Precision: 0.7567
Средний Recall: 0.7938
Средний F1-Score: 0.7596
🏃 View run 66 at: http://127.0.0.1:5000/#/experiments/1/runs/29d1b74677404bdeb166a4878a7ef312
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 67
Best params: {'n_estimators': 800, 'max_depth': 21, 'min_samples_split': 3, 'min_samples_leaf': 2}


In [58]:
study.best_trial.user_attrs['pipeline_params']

{'model__n_estimators': 800,
 'model__max_depth': 21,
 'model__min_samples_split': 3,
 'model__min_samples_leaf': 2}

In [59]:
pipeline_rf_best_optuna = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('numeric_scaling', StandardScaler(), numeric_features),
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ])),
        ('model', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'))
    ])

pipeline_rf_best_optuna.set_params(**study.best_trial.user_attrs['pipeline_params'])
pipeline_rf_best_optuna.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric_scaling', ...), ('categorical_encoding', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of

In [60]:
y_pred_proba = pipeline_rf_best_optuna.predict_proba(X_test)

df_test = df.loc[X_test.index].copy()
df_test['y_pred_proba'] = y_pred_proba[:, 1]

In [61]:
ndcg_rf, precision_rf, recall_rf, f1_rf = calculate_metrics(df_test)

metrics_rf = {
    'NDCG': ndcg_rf,
    'Precision': precision_rf,
    'Recall': recall_rf,
    'F1': f1_rf
}

Средний NDCG: 0.7780
Средний Precision: 0.6921
Средний Recall: 0.7265
Средний F1-Score: 0.6956


In [62]:
RUN_NAME = "best_optuna_rf" 
REGISTRY_MODEL_NAME = "best_optuna_rf"

In [63]:
signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id

    lr_info = mlflow.sklearn.log_model(sk_model=pipeline_rf_best_optuna, 
                                       artifact_path='best_optuna_rf',
                                       registered_model_name=REGISTRY_MODEL_NAME,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_rf)
    mlflow.log_params(best_params_optuna)

Registered model 'best_optuna_rf' already exists. Creating a new version of this model...
2026/05/04 02:09:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_rf, version 6


🏃 View run best_optuna_rf at: http://127.0.0.1:5000/#/experiments/1/runs/110fa1ca7271436ea98f6b4338c7bbe4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '6' of model 'best_optuna_rf'.


# XGBClassifier

In [64]:
def objective_xgb(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    pipeline_xgb = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE, 
            eval_metric='logloss', 
            use_label_encoder=False, 
            verbosity=0,
            scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
        ))
    ])
    
    pipeline_xgb.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_xgb.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_xgb.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [65]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE_XGB = 'XGBClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_XGB, experiment_id=experiment_id) as run:
    run_id_xgb = run.info.run_id

🏃 View run XGBClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/75c7801f25b7489a8131dfad7498afd7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [66]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME_XGB = "XGBClassifier_optuna"

mlflc_xgb = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_xgb}}
)

In [67]:
study_xgb = optuna.create_study(direction='maximize', 
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                study_name=STUDY_NAME_XGB,
                                storage=STUDY_DB_NAME,
                                load_if_exists=True)

study_xgb.optimize(objective_xgb, 
                   n_trials=10, 
                   callbacks=[mlflc_xgb]
                  )

best_params_xgb = study_xgb.best_params
print(f"Number of finished trials: {len(study_xgb.trials)}")
print(f"Best params XGBoost: {best_params_xgb}")

[I 2026-05-04 02:09:03,810] Using an existing study with name 'XGBClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8539
Средний Precision: 0.6623
Средний Recall: 0.8332
Средний F1-Score: 0.7141
Средний NDCG: 0.8433
Средний Precision: 0.6537
Средний Recall: 0.8212
Средний F1-Score: 0.7034


[I 2026-05-04 02:09:17,955] Trial 50 finished with value: 0.8510896371698593 and parameters: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.04957493265622066, 'min_child_weight': 10}. Best is trial 21 with value: 0.8583336583209565.


Средний NDCG: 0.8511
Средний Precision: 0.6596
Средний Recall: 0.8261
Средний F1-Score: 0.7099
🏃 View run 50 at: http://127.0.0.1:5000/#/experiments/1/runs/6d11c7078e0c47b3a782282aedb61c22
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8608
Средний Precision: 0.7539
Средний Recall: 0.8217
Средний F1-Score: 0.7703
Средний NDCG: 0.8476
Средний Precision: 0.7446
Средний Recall: 0.8056
Средний F1-Score: 0.7578


[I 2026-05-04 02:26:52,259] Trial 51 finished with value: 0.8569878956677255 and parameters: {'n_estimators': 400, 'max_depth': 10, 'learning_rate': 0.10445200686210339, 'min_child_weight': 9}. Best is trial 21 with value: 0.8583336583209565.


...

Средний NDCG: 0.8573
Средний Precision: 0.7583
Средний Recall: 0.8062
Средний F1-Score: 0.7661
🏃 View run 57 at: http://127.0.0.1:5000/#/experiments/1/runs/7dc24caa60a64813aee16a6af4091ee8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8608
Средний Precision: 0.7230
Средний Recall: 0.8308
Средний F1-Score: 0.7546
Средний NDCG: 0.8485
Средний Precision: 0.7186
Средний Recall: 0.8205
Средний F1-Score: 0.7486


[I 2026-05-04 02:45:21,970] Trial 58 finished with value: 0.857180772858686 and parameters: {'n_estimators': 350, 'max_depth': 6, 'learning_rate': 0.15921992938010293, 'min_child_weight': 4}. Best is trial 21 with value: 0.8583336583209565.


Средний NDCG: 0.8572
Средний Precision: 0.7306
Средний Recall: 0.8267
Средний F1-Score: 0.7580
🏃 View run 58 at: http://127.0.0.1:5000/#/experiments/1/runs/3829291079044ab68fcf071ab5e69303
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8620
Средний Precision: 0.7412
Средний Recall: 0.8308
Средний F1-Score: 0.7662
Средний NDCG: 0.8481
Средний Precision: 0.7308
Средний Recall: 0.8159
Средний F1-Score: 0.7542


[I 2026-05-04 02:45:39,410] Trial 59 finished with value: 0.8576425206864827 and parameters: {'n_estimators': 800, 'max_depth': 5, 'learning_rate': 0.18329471317629709, 'min_child_weight': 5}. Best is trial 21 with value: 0.8583336583209565.


Средний NDCG: 0.8576
Средний Precision: 0.7402
Средний Recall: 0.8229
Средний F1-Score: 0.7629
🏃 View run 59 at: http://127.0.0.1:5000/#/experiments/1/runs/edd622b1373a409abeb09366141504fd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 60
Best params XGBoost: {'n_estimators': 600, 'max_depth': 10, 'learning_rate': 0.09866580252678012, 'min_child_weight': 10}


In [68]:
pipeline_xgb_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', XGBClassifier(
        random_state=RANDOM_STATE, 
        eval_metric='logloss', 
        use_label_encoder=False, 
        verbosity=0,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
    ))
])

pipeline_xgb_best.set_params(**study_xgb.best_trial.user_attrs['pipeline_params'])
pipeline_xgb_best.fit(X_train, y_train)

y_pred_proba_xgb = pipeline_xgb_best.predict_proba(X_test)

df_test_xgb = df.loc[X_test.index].copy()
df_test_xgb['y_pred_proba'] = y_pred_proba_xgb[:, 1]

ndcg_xgb, precision_xgb, recall_xgb, f1_xgb = calculate_metrics(df_test_xgb)
metrics_xgb = {
    'NDCG': ndcg_xgb,
    'Precision': precision_xgb,
    'Recall': recall_xgb,
    'F1': f1_xgb
}

Средний NDCG: 0.7792
Средний Precision: 0.6968
Средний Recall: 0.7411
Средний F1-Score: 0.7054


In [69]:
RUN_NAME_XGB = "best_optuna_xgb"
REGISTRY_MODEL_NAME_XGB = "best_optuna_xgb"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_XGB, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    xgb_info = mlflow.sklearn.log_model(sk_model=pipeline_xgb_best, 
                                       artifact_path='best_optuna_xgb',
                                       registered_model_name=REGISTRY_MODEL_NAME_XGB,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_xgb)
    mlflow.log_params(best_params_xgb)

Registered model 'best_optuna_xgb' already exists. Creating a new version of this model...
2026/05/04 02:45:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_xgb, version 7


🏃 View run best_optuna_xgb at: http://127.0.0.1:5000/#/experiments/1/runs/f92e46692a6f406e94c64d8a8c75c306
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '7' of model 'best_optuna_xgb'.


# LGBMClassifier

In [70]:
def objective_lgbm(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'model__min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }
    
    pipeline_lgbm = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', LGBMClassifier(
            random_state=RANDOM_STATE, 
            verbose=-1,
            class_weight='balanced'
        ))
    ])
    
    pipeline_lgbm.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lgbm.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lgbm.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [71]:
RUN_NAME_OPTUNE_LGBM = 'LGBMClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_LGBM, experiment_id=experiment_id) as run:
    run_id_lgbm = run.info.run_id

🏃 View run LGBMClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/bb68cdbd5a224b6eaccf4bd4bf9a3364
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [72]:
STUDY_NAME_LGBM = "LGBMClassifier_optuna"

mlflc_lgbm = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_lgbm}}
)

In [73]:
study_lgbm = optuna.create_study(direction='maximize', 
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 study_name=STUDY_NAME_LGBM,
                                 storage=STUDY_DB_NAME,
                                 load_if_exists=True)

study_lgbm.optimize(objective_lgbm, 
                    n_trials=10, 
                    callbacks=[mlflc_lgbm]
                   )

best_params_lgbm = study_lgbm.best_params
print(f"Number of finished trials: {len(study_lgbm.trials)}")
print(f"Best params LGBM: {best_params_lgbm}")

[I 2026-05-04 02:45:48,161] Using an existing study with name 'LGBMClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8602
Средний Precision: 0.7735
Средний Recall: 0.8038
Средний F1-Score: 0.7733
Средний NDCG: 0.8477
Средний Precision: 0.7655
Средний Recall: 0.7906
Средний F1-Score: 0.7635


[I 2026-05-04 02:46:56,222] Trial 50 finished with value: 0.856016744464436 and parameters: {'n_estimators': 800, 'max_depth': 13, 'learning_rate': 0.133984997741435, 'num_leaves': 90, 'min_child_samples': 35}. Best is trial 43 with value: 0.8587410189011325.


Средний NDCG: 0.8560
Средний Precision: 0.7697
Средний Recall: 0.7920
Средний F1-Score: 0.7658
🏃 View run 50 at: http://127.0.0.1:5000/#/experiments/1/runs/2994432738ce4f5880eef5ea6bbd36c1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8626
Средний Precision: 0.7488
Средний Recall: 0.8265
Средний F1-Score: 0.7689
Средний NDCG: 0.8497
Средний Precision: 0.7383
Средний Recall: 0.8138
Средний F1-Score: 0.7581


[I 2026-05-04 02:48:27,946] Trial 51 finished with value: 0.8585027468006385 and parameters: {'n_estimators': 950, 'max_depth': 14, 'learning_rate': 0.030744527638479742, 'num_leaves': 108, 'min_child_samples': 77}. Best is trial 43 with value: 0.8587410189011325.


...

Средний NDCG: 0.8570
Средний Precision: 0.7164
Средний Recall: 0.8276
Средний F1-Score: 0.7493
🏃 View run 57 at: http://127.0.0.1:5000/#/experiments/1/runs/d72bac8e11194206aabbf9e3b5fc1b2f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8618
Средний Precision: 0.7386
Средний Recall: 0.8308
Средний F1-Score: 0.7644
Средний NDCG: 0.8489
Средний Precision: 0.7289
Средний Recall: 0.8178
Средний F1-Score: 0.7535


[I 2026-05-04 02:57:37,756] Trial 58 finished with value: 0.8567359237505543 and parameters: {'n_estimators': 850, 'max_depth': 6, 'learning_rate': 0.08790770744930856, 'num_leaves': 63, 'min_child_samples': 74}. Best is trial 43 with value: 0.8587410189011325.


Средний NDCG: 0.8567
Средний Precision: 0.7357
Средний Recall: 0.8224
Средний F1-Score: 0.7597
🏃 View run 58 at: http://127.0.0.1:5000/#/experiments/1/runs/6652fe791a69485a9593d6e8efdf76e2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8625
Средний Precision: 0.7505
Средний Recall: 0.8259
Средний F1-Score: 0.7695
Средний NDCG: 0.8495
Средний Precision: 0.7375
Средний Recall: 0.8120
Средний F1-Score: 0.7569


[I 2026-05-04 02:59:15,723] Trial 59 finished with value: 0.8569739564495952 and parameters: {'n_estimators': 900, 'max_depth': 12, 'learning_rate': 0.029817362669024915, 'num_leaves': 129, 'min_child_samples': 67}. Best is trial 43 with value: 0.8587410189011325.


Средний NDCG: 0.8570
Средний Precision: 0.7489
Средний Recall: 0.8173
Средний F1-Score: 0.7659
🏃 View run 59 at: http://127.0.0.1:5000/#/experiments/1/runs/79ccfa87655c48cdb15c313784e2697f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 60
Best params LGBM: {'n_estimators': 900, 'max_depth': 14, 'learning_rate': 0.05651940848512141, 'num_leaves': 72, 'min_child_samples': 88}


In [74]:
pipeline_lgbm_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', LGBMClassifier(
        random_state=RANDOM_STATE, 
        verbose=-1,
        class_weight='balanced'
    ))
])

pipeline_lgbm_best.set_params(**study_lgbm.best_trial.user_attrs['pipeline_params'])
pipeline_lgbm_best.fit(X_train, y_train)

y_pred_proba_lgbm = pipeline_lgbm_best.predict_proba(X_test)

df_test_lgbm = df.loc[X_test.index].copy()
df_test_lgbm['y_pred_proba'] = y_pred_proba_lgbm[:, 1]

ndcg_lgbm, precision_lgbm, recall_lgbm, f1_lgbm = calculate_metrics(df_test_lgbm)
metrics_lgbm = {
    'NDCG': ndcg_lgbm,
    'Precision': precision_lgbm,
    'Recall': recall_lgbm,
    'F1': f1_lgbm
}

Средний NDCG: 0.7805
Средний Precision: 0.6846
Средний Recall: 0.7486
Средний F1-Score: 0.7008


In [75]:
RUN_NAME_LGBM = "best_optuna_lgbm"
REGISTRY_MODEL_NAME_LGBM = "best_optuna_lgbm"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_LGBM, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    lgbm_info = mlflow.sklearn.log_model(sk_model=pipeline_lgbm_best, 
                                        artifact_path='best_optuna_lgbm',
                                        registered_model_name=REGISTRY_MODEL_NAME_LGBM,
                                        input_example=input_example,
                                        code_paths=code_paths,
                                        await_registration_for=60
                                       )
    mlflow.log_metrics(metrics_lgbm)
    mlflow.log_params(best_params_lgbm)

Registered model 'best_optuna_lgbm' already exists. Creating a new version of this model...
2026/05/04 02:59:39 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lgbm, version 7


🏃 View run best_optuna_lgbm at: http://127.0.0.1:5000/#/experiments/1/runs/6706957f04ae45d68735dcd2d3a767f4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '7' of model 'best_optuna_lgbm'.


# CatBoostClassifier

In [76]:
def objective_catboost(trial: optuna.Trial) -> float:
    params = {
        'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'model__depth': trial.suggest_int('depth', 3, 10),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
    }
    
    pipeline_catboost = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_catboost.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [77]:
RUN_NAME_OPTUNE_CATBOOST = 'CatBoostClassifier_optuna'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
    run_id_catboost = run.info.run_id

🏃 View run CatBoostClassifier_optuna at: http://127.0.0.1:5000/#/experiments/1/runs/94861f3e2fe94e25a9fd2b9feaa18ff4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [78]:
STUDY_NAME_CATBOOST = "CatBoostClassifier_optuna"

mlflc_catboost = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
)

In [79]:
study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=True)

study_catboost.optimize(objective_catboost, 
                        n_trials=10, 
                        callbacks=[mlflc_catboost]
                       )

best_params_catboost = study_catboost.best_params
print(f"Number of finished trials: {len(study_catboost.trials)}")
print(f"Best params CatBoost: {best_params_catboost}")

[I 2026-05-04 02:59:39,274] Using an existing study with name 'CatBoostClassifier_optuna' instead of creating a new one.


Средний NDCG: 0.8614
Средний Precision: 0.7466
Средний Recall: 0.8272
Средний F1-Score: 0.7681
Средний NDCG: 0.8485
Средний Precision: 0.7363
Средний Recall: 0.8138
Средний F1-Score: 0.7564


[I 2026-05-04 03:00:04,927] Trial 51 finished with value: 0.857421824044987 and parameters: {'iterations': 850, 'depth': 8, 'learning_rate': 0.2232530529408405, 'l2_leaf_reg': 4.167477662688214}. Best is trial 45 with value: 0.8584356136153125.


Средний NDCG: 0.8574
Средний Precision: 0.7492
Средний Recall: 0.8159
Средний F1-Score: 0.7656
🏃 View run 51 at: http://127.0.0.1:5000/#/experiments/1/runs/5a90a6bea52b4fa88be253cbb881b6d3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8623
Средний Precision: 0.7397
Средний Recall: 0.8290
Средний F1-Score: 0.7648
Средний NDCG: 0.8495
Средний Precision: 0.7382
Средний Recall: 0.8133
Средний F1-Score: 0.7577


[I 2026-05-04 03:00:35,521] Trial 52 finished with value: 0.8576677175047919 and parameters: {'iterations': 900, 'depth': 9, 'learning_rate': 0.0987029110568357, 'l2_leaf_reg': 3.043486634995781}. Best is trial 45 with value: 0.8584356136153125.


...

Средний NDCG: 0.8567
Средний Precision: 0.7039
Средний Recall: 0.8308
Средний F1-Score: 0.7424
🏃 View run 58 at: http://127.0.0.1:5000/#/experiments/1/runs/771463a4225146f98bba50c9896c7b1d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8616
Средний Precision: 0.7530
Средний Recall: 0.8271
Средний F1-Score: 0.7718
Средний NDCG: 0.8489
Средний Precision: 0.7407
Средний Recall: 0.8113
Средний F1-Score: 0.7584


[I 2026-05-04 03:03:53,000] Trial 59 finished with value: 0.8574835420501067 and parameters: {'iterations': 950, 'depth': 9, 'learning_rate': 0.10389590734703245, 'l2_leaf_reg': 3.165014806633651}. Best is trial 45 with value: 0.8584356136153125.


Средний NDCG: 0.8575
Средний Precision: 0.7462
Средний Recall: 0.8145
Средний F1-Score: 0.7632
🏃 View run 59 at: http://127.0.0.1:5000/#/experiments/1/runs/0729a09975c04482b38baaba1b004e03
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8607
Средний Precision: 0.7641
Средний Recall: 0.8139
Средний F1-Score: 0.7729
Средний NDCG: 0.8478
Средний Precision: 0.7517
Средний Recall: 0.8022
Средний F1-Score: 0.7605


[I 2026-05-04 03:04:31,747] Trial 60 finished with value: 0.8565998424191699 and parameters: {'iterations': 900, 'depth': 10, 'learning_rate': 0.14794015317955714, 'l2_leaf_reg': 1.6234145278950396}. Best is trial 45 with value: 0.8584356136153125.


Средний NDCG: 0.8566
Средний Precision: 0.7604
Средний Recall: 0.8032
Средний F1-Score: 0.7658
🏃 View run 60 at: http://127.0.0.1:5000/#/experiments/1/runs/f683ac0f76824aeca2286614bf020e2a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 61
Best params CatBoost: {'iterations': 900, 'depth': 8, 'learning_rate': 0.09987315180281384, 'l2_leaf_reg': 2.020097979184651}


In [80]:
pipeline_catboost_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', CatBoostClassifier(
        random_state=RANDOM_STATE, 
        verbose=0, 
        auto_class_weights='Balanced'
    ))
])

pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
pipeline_catboost_best.fit(X_train, y_train)

y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_test)

df_test_catboost = df.loc[X_test.index].copy()
df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]

ndcg_catboost, precision_catboost, recall_catboost, f1_catboost = calculate_metrics(df_test_catboost)
metrics_catboost = {
    'NDCG': ndcg_catboost,
    'Precision': precision_catboost,
    'Recall': recall_catboost,
    'F1': f1_catboost
}

Средний NDCG: 0.7798
Средний Precision: 0.6762
Средний Recall: 0.7518
Средний F1-Score: 0.6968


In [81]:
RUN_NAME_CATBOOST = "best_optuna_catboost"
REGISTRY_MODEL_NAME_CATBOOST = "best_optuna_catboost"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                             artifact_path='best_optuna_catboost',
                                             registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                             input_example=input_example,
                                             code_paths=code_paths,
                                             await_registration_for=60
                                            )
    mlflow.log_metrics(metrics_catboost)
    mlflow.log_params(best_params_catboost)

Registered model 'best_optuna_catboost' already exists. Creating a new version of this model...
2026/05/04 03:04:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_catboost, version 8


🏃 View run best_optuna_catboost at: http://127.0.0.1:5000/#/experiments/1/runs/e5001f8f303648d39aebeeb93728e268
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '8' of model 'best_optuna_catboost'.


# Model comparison

In [82]:
models_comparison = pd.DataFrame({
    'Model': ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost'],
    'NDCG': [ndcg_lr, ndcg_rf, 
             ndcg_xgb, ndcg_lgbm, ndcg_catboost],
    'Precision': [precision_lr, precision_rf, 
                  precision_xgb, precision_lgbm, precision_catboost],
    'Recall': [recall_lr, recall_rf, 
               recall_xgb, recall_lgbm, recall_catboost],
    'F1': [f1_lr, f1_rf, 
           f1_xgb, f1_lgbm, f1_catboost]
})

models_comparison

,Model,NDCG,Precision,Recall,F1
0,LogisticRegression,0.751483,0.638633,0.598481,0.602124
1,RandomForest,0.778008,0.692051,0.726508,0.695558
2,XGBoost,0.779192,0.696847,0.741136,0.705449
3,LightGBM,0.780483,0.684553,0.748602,0.700788
4,CatBoost,0.779769,0.676177,0.751765,0.696813


In [83]:
best_model_idx = models_comparison['NDCG'].idxmax()
print(f"\nЛучшая модель: {models_comparison.iloc[best_model_idx]['Model']} с NDCG = {models_comparison.iloc[best_model_idx]['NDCG']:.4f}")


Лучшая модель: LightGBM с NDCG = 0.7805


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

По итогам подбора гиперпараметров лучшее качество показала модель `XGBClassifier`. Теперь попробуем добавить дополнительные признаки от кандидата генераторов. В первую очередь используем коллаборативную фильтрацию.

</div>

# ALS

## Feature engineering

In [84]:
def create_interaction_matrix(df):
    unique_vacancies = df['vacancy_id'].unique().tolist()
    unique_resumes = df['resume_id'].unique().tolist()

    id2vacancy = dict(enumerate(unique_vacancies))
    id2resume = dict(enumerate(unique_resumes))
    vacancy2id = {v: k for k, v in id2vacancy.items()}
    resume2id = {v: k for k, v in id2resume.items()}

    rows = [vacancy2id[vacancy] for vacancy in df['vacancy_id']]
    cols = [resume2id[resume] for resume in df['resume_id']]

    interactions_matrix = csr_matrix(
        (df['target'], (rows, cols)),
        shape=(len(unique_vacancies), len(unique_resumes)),
        dtype=np.float32,
    )

    return interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes

In [85]:
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(df)

print(f"Размерность матрицы взаимодействий: {interactions_matrix.shape}")
print(f"Плотность матрицы: {interactions_matrix.nnz / (interactions_matrix.shape[0] * interactions_matrix.shape[1]):.6f}")

Размерность матрицы взаимодействий: (3409, 20130)
Плотность матрицы: 0.004687


In [86]:
def get_factors(interactions_matrix):
    als_model = AlternatingLeastSquares(
        factors=64,          
        regularization=0.1,  
        iterations=30,       
        random_state=RANDOM_STATE,
        num_threads=0
    )
    
    als_model.fit(interactions_matrix.T)

    vacancy_factors = als_model.item_factors
    resume_factors = als_model.user_factors

    return vacancy_factors, resume_factors

In [87]:
vacancy_factors, resume_factors = get_factors(interactions_matrix)

print(f"Размерность эмбеддингов вакансий: {vacancy_factors.shape}")
print(f"Размерность эмбеддингов резюме: {resume_factors.shape}")

  0%|          | 0/30 [00:00<?, ?it/s]

Размерность эмбеддингов вакансий: (3409, 64)
Размерность эмбеддингов резюме: (20130, 64)


In [88]:
def get_als_score(vacancy_id, resume_id):
    if vacancy_id not in vacancy2id or resume_id not in resume2id:
        return 0
    else:
        vac_idx = vacancy2id[vacancy_id]
        res_idx = resume2id[resume_id]
        score = np.dot(vacancy_factors[vac_idx], resume_factors[res_idx])
        return score

df['als_score'] = df.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

In [89]:
df[['vacancy_id', 'resume_id', 'target', 'als_score']].head()

,vacancy_id,resume_id,target,als_score
0,126167948,6969174,1,0.000042
1,126167948,9100077,1,0.000033
2,126167948,32644957,1,0.000019
3,126167948,27220466,1,0.000021
4,126167948,7532708,1,0.000021


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Проверим схожесть резюме по латентным векторам.

</div>

In [90]:
def find_similar_resumes(resume_id, resume_factors, n_similar=10, metric='cosine'):
    """
    Находит N наиболее похожих резюме на заданное.
    """
    if resume_id not in resume2id:
        print(f"Резюме с ID {resume_id} не найдено")
        return []

    target_idx = resume2id[resume_id]
    target_vector = resume_factors[target_idx].reshape(1, -1)
    
    if metric == 'cosine':
        similarities = cosine_similarity(target_vector, resume_factors)[0]
    else:
        similarities = np.dot(resume_factors, target_vector.T).flatten()
    
    similar_indices = np.argsort(similarities)[::-1][1:n_similar+1]
    
    similar_resumes = []
    for idx in similar_indices:
        sim_resume_id = unique_resumes[idx]
        similarity_score = similarities[idx]
        similar_resumes.append((sim_resume_id, similarity_score))
    
    return similar_resumes

def get_resume_profile(resume_id, df):
    """Вспомогательная функция для получения информации о резюме"""
    resume_data = df[df['resume_id'] == resume_id].iloc[0]
    return {
        'resume_id': resume_id,
        'title': resume_data.get('resume_title', 'N/A'),
        'specialization': resume_data.get('resume_specialization', 'N/A'),
        'skills': resume_data.get('resume_skills', 'N/A'),
        'experience': resume_data.get('resume_total_experience', 'N/A'),
        'salary': resume_data.get('resume_salary', 'N/A'),
        'location': resume_data.get('resume_location', 'N/A')
    }

In [91]:
example_resume_id = df['resume_id'].sample(1, random_state=RANDOM_STATE).values[0].item()
print(f"Исходное резюме (ID: {example_resume_id}):")
original_profile = get_resume_profile(example_resume_id, df)
for key, value in original_profile.items():
    print(f"  {key}: {value}")

print(f"\nТоп-5 похожих резюме (косинусная близость):")
similar_resumes = find_similar_resumes(example_resume_id, resume_factors, n_similar=5, metric='cosine')

for i, (sim_id, score) in enumerate(similar_resumes, 1):
    profile = get_resume_profile(sim_id, df)
    print(f"\n{i}. Резюме ID: {sim_id} (сходство: {score:.4f})")
    print(f"   Должность: {profile['title']}")
    print(f"   Специализация: {profile['specialization']}")
    print(f"   Локация: {profile['location']}")
    print(f"   Опыт: {profile['experience']}")

Исходное резюме (ID: 65353364):
  resume_id: 65353364
  title: Системный администратор
  specialization: ['Сетевой инженер', 'Системный администратор', 'Системный инженер']
  skills: [' Linux', 'Основы анализа данных.', 'Основы Python', 'Информационные технологии', 'Администрирование сетевого оборудования', 'Сетевые технологии', 'Active Directory', 'VLAN', 'OSPF', 'Системное администрирование', 'Техническая поддержка', 'Cisco', 'Qtech', 'Huawei', 'Eltex', 'Основы SQL', 'Helpdesk', 'Cloud', 'OSI', 'Администрирование серверов Linux', 'Zabbix']
  experience: 15 лет 10 месяцев
  salary: 170000.0
  location: Москва

Топ-5 похожих резюме (косинусная близость):

1. Резюме ID: 55976730 (сходство: 0.0000)
   Должность: Инженер-программист .Net/C#
   Специализация: ['Программист, разработчик']
   Локация: Москва
   Опыт: 5 лет 9 месяцев

2. Резюме ID: 37812953 (сходство: 0.0000)
   Должность: Joomla Developer
   Специализация: ['Программист, разработчик']
   Локация: Донецк
   Опыт: 12 лет 7 мес

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Разделим тренировочную выборку еще на две, чтобы избежать подглядывания и добавить в признак `als_score` нулевые значения в тренировочный датасет, т.к. у нас могут быть вакансии и резюме без взаимодействий до обучения.

</div>

In [92]:
X_train_als, X_test_als, y_train_als, y_test_als = train_test_split(X_train, y_train, test_size=0.3, random_state=RANDOM_STATE, stratify=y_train)

In [93]:
train_als_interactions = df.loc[X_train_als.index, ['vacancy_id', 'resume_id', 'target']]

interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(train_als_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

# возвращаем айдишники вакансий и резюме для расчета als score
X_train[['vacancy_id', 'resume_id']] = df.loc[X_train.index, ['vacancy_id', 'resume_id']]
X_train['als_score'] = X_train.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

X_train = X_train.drop(['vacancy_id', 'resume_id'], axis=1)

  0%|          | 0/30 [00:00<?, ?it/s]

In [94]:
# проверим есть ли вакансии и резюме без взаимодействий до обучения
X_train[X_train['als_score'] == 0]

,vacancy_area,vacancy_experience,vacancy_employment,vacancy_schedule,resume_salary,resume_age,resume_experience_months,resume_location,resume_gender,resume_applicant_status,resume_last_company_experience_months,location_matching,resume_skill_count_in_vacancy,last_position_in_vacancy,similarity_score_tfidf,als_score
286115,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,40000.0,38.0,223.0,Москва,Неизвестно,NDT,208.0,1,0,0.25,0.015081,0.0
216165,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.038536,0.0
297591,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.016496,0.0
192099,Москва,Более 6 лет,Полная занятость,Полный день,30000.0,57.0,289.0,Москва,Мужчина,NDT,170.0,1,0,0.00,0.000000,0.0
82523,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,0.0,38.0,185.0,Москва,Мужчина,NDT,114.0,1,4,0.00,0.063418,0.0
32590,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,0.0,38.0,185.0,Москва,Мужчина,NDT,114.0,1,2,0.00,0.101719,0.0
260861,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,30000.0,57.0,289.0,Москва,Мужчина,NDT,170.0,1,0,0.00,0.020658,0.0
52865,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.011811,0.0
175258,Москва,От 1 года до 3 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.021698,0.0
217714,Москва,От 3 до 6 лет,Полная занятость,Удаленная работа,150000.0,35.0,152.0,Москва,Мужчина,Активно ищет работу,89.0,1,0,0.00,0.012118,0.0


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Теперь добавим признак `als_score` для тестовой выборке, но уже возьмем матрицу интеракций на полной тренировочной выборке.

</div>

In [95]:
train_interactions = df.loc[X_train.index, ['vacancy_id', 'resume_id', 'target']]

interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes = create_interaction_matrix(train_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

# возвращаем айдишники вакансий и резюме для расчета als score
X_test[['vacancy_id', 'resume_id']] = df.loc[X_test.index, ['vacancy_id', 'resume_id']]
X_test['als_score'] = X_test.apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), 
    axis=1
)

X_test = X_test.drop(['vacancy_id', 'resume_id'], axis=1)

  0%|          | 0/30 [00:00<?, ?it/s]

<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Выборки получены, обучим `XGBClassifier` с подбором гиперпараметров.

</div>

In [129]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 257313 entries, 188418 to 283280
Data columns (total 16 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   vacancy_area                           257313 non-null  object 
 1   vacancy_experience                     257313 non-null  object 
 2   vacancy_employment                     257313 non-null  object 
 3   vacancy_schedule                       257313 non-null  object 
 4   resume_salary                          257313 non-null  float64
 5   resume_age                             257313 non-null  float64
 6   resume_experience_months               257313 non-null  float64
 7   resume_location                        257313 non-null  object 
 8   resume_gender                          257313 non-null  object 
 9   resume_applicant_status                257313 non-null  object 
 10  resume_last_company_experience_months  257313 non-null  

## XGBClassifier

In [96]:
def objective_xgb(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }
    
    pipeline_xgb = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', XGBClassifier(
            random_state=RANDOM_STATE, 
            eval_metric='logloss', 
            use_label_encoder=False, 
            verbosity=0,
            scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
        ))
    ])
    
    pipeline_xgb.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_xgb.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_xgb.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [97]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

RUN_NAME_OPTUNE_XGB = 'XGBClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_XGB, experiment_id=experiment_id) as run:
    run_id_xgb = run.info.run_id

🏃 View run XGBClassifier_optuna_als at: http://127.0.0.1:5000/#/experiments/1/runs/937ace3a95424921a80fe2571e3f3abd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [98]:
STUDY_DB_NAME = "sqlite:///local.study.db"
STUDY_NAME_XGB = "XGBClassifier_optuna_als"

mlflc_xgb = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_xgb}}
)

In [99]:
study_xgb = optuna.create_study(direction='maximize', 
                                sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                study_name=STUDY_NAME_XGB,
                                storage=STUDY_DB_NAME,
                                load_if_exists=True)

study_xgb.optimize(objective_xgb, 
                   n_trials=10, 
                   callbacks=[mlflc_xgb]
                  )

best_params_xgb = study_xgb.best_params
print(f"Number of finished trials: {len(study_xgb.trials)}")
print(f"Best params XGBoost: {best_params_xgb}")

[I 2026-05-04 03:04:53,270] Using an existing study with name 'XGBClassifier_optuna_als' instead of creating a new one.


Средний NDCG: 0.8660
Средний Precision: 0.7260
Средний Recall: 0.8522
Средний F1-Score: 0.7677
Средний NDCG: 0.8566
Средний Precision: 0.7180
Средний Recall: 0.8414
Средний F1-Score: 0.7586


[I 2026-05-04 03:05:08,421] Trial 50 finished with value: 0.8605046402672311 and parameters: {'n_estimators': 350, 'max_depth': 5, 'learning_rate': 0.010260022325256595, 'min_child_weight': 2}. Best is trial 32 with value: 0.8638529990938331.


Средний NDCG: 0.8605
Средний Precision: 0.7247
Средний Recall: 0.8436
Средний F1-Score: 0.7637
🏃 View run 50 at: http://127.0.0.1:5000/#/experiments/1/runs/da59bafce41546ea95280c6ac70da5b1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8680
Средний Precision: 0.7706
Средний Recall: 0.8474
Средний F1-Score: 0.7947
Средний NDCG: 0.8583
Средний Precision: 0.7632
Средний Recall: 0.8392
Средний F1-Score: 0.7869


[I 2026-05-04 03:05:25,312] Trial 51 finished with value: 0.8629802251670222 and parameters: {'n_estimators': 450, 'max_depth': 7, 'learning_rate': 0.032334427983478194, 'min_child_weight': 1}. Best is trial 32 with value: 0.8638529990938331.


...

Средний NDCG: 0.8630
Средний Precision: 0.7778
Средний Recall: 0.8382
Средний F1-Score: 0.7952
🏃 View run 57 at: http://127.0.0.1:5000/#/experiments/1/runs/c93d286bc9bd416a80c1746f80a58d14
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8681
Средний Precision: 0.7355
Средний Recall: 0.8535
Средний F1-Score: 0.7746
Средний NDCG: 0.8580
Средний Precision: 0.7319
Средний Recall: 0.8439
Средний F1-Score: 0.7690


[I 2026-05-04 03:07:39,257] Trial 58 finished with value: 0.8627850471631034 and parameters: {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.021011482173402783, 'min_child_weight': 3}. Best is trial 32 with value: 0.8638529990938331.


Средний NDCG: 0.8628
Средний Precision: 0.7378
Средний Recall: 0.8488
Средний F1-Score: 0.7748
🏃 View run 58 at: http://127.0.0.1:5000/#/experiments/1/runs/774cbfcdaca445a9b711ecd20e43b483
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8675
Средний Precision: 0.7995
Средний Recall: 0.8314
Средний F1-Score: 0.8055
Средний NDCG: 0.8578
Средний Precision: 0.7914
Средний Recall: 0.8236
Средний F1-Score: 0.7971


[I 2026-05-04 03:07:59,355] Trial 59 finished with value: 0.8635383544831327 and parameters: {'n_estimators': 650, 'max_depth': 9, 'learning_rate': 0.05723129341128079, 'min_child_weight': 4}. Best is trial 32 with value: 0.8638529990938331.


Средний NDCG: 0.8635
Средний Precision: 0.7946
Средний Recall: 0.8282
Средний F1-Score: 0.8008
🏃 View run 59 at: http://127.0.0.1:5000/#/experiments/1/runs/1d3d01e0330545208dbbde5293eb4f4b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 60
Best params XGBoost: {'n_estimators': 600, 'max_depth': 9, 'learning_rate': 0.0482381453391051, 'min_child_weight': 5}


In [100]:
pipeline_xgb_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', XGBClassifier(
        random_state=RANDOM_STATE, 
        eval_metric='logloss', 
        use_label_encoder=False, 
        verbosity=0,
        scale_pos_weight=(len(y_train) - sum(y_train)) / sum(y_train)
    ))
])

pipeline_xgb_best.set_params(**study_xgb.best_trial.user_attrs['pipeline_params'])
pipeline_xgb_best.fit(X_train, y_train)

y_pred_proba_xgb = pipeline_xgb_best.predict_proba(X_test)

df_test_xgb = df.loc[X_test.index].copy()
df_test_xgb['y_pred_proba'] = y_pred_proba_xgb[:, 1]

ndcg_xgb_als, precision_xgb_als, recall_xgb_als, f1_xgb_als = calculate_metrics(df_test_xgb)
metrics_xgb_als = {
    'NDCG': ndcg_xgb_als,
    'Precision': precision_xgb_als,
    'Recall': recall_xgb_als,
    'F1': f1_xgb_als
}

Средний NDCG: 0.7812
Средний Precision: 0.7086
Средний Recall: 0.7425
Средний F1-Score: 0.7137


In [101]:
RUN_NAME_XGB = "best_optuna_xgb_als"
REGISTRY_MODEL_NAME_XGB = "best_optuna_xgb_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_XGB, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    xgb_info = mlflow.sklearn.log_model(sk_model=pipeline_xgb_best, 
                                       artifact_path='best_optuna_xgb_als',
                                       registered_model_name=REGISTRY_MODEL_NAME_XGB,
                                       input_example=input_example,
                                       code_paths=code_paths,
                                       await_registration_for=60
                                      )
    mlflow.log_metrics(metrics_xgb_als)
    mlflow.log_params(best_params_xgb)

Registered model 'best_optuna_xgb_als' already exists. Creating a new version of this model...
2026/05/04 03:08:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_xgb_als, version 6


🏃 View run best_optuna_xgb_als at: http://127.0.0.1:5000/#/experiments/1/runs/b4b4248a00434aa287c1f2177b2bdbbc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '6' of model 'best_optuna_xgb_als'.


## LGBMClassifier

In [102]:
def objective_lgbm(trial: optuna.Trial) -> float:
    params = {
        'model__n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=50),
        'model__max_depth': trial.suggest_int('max_depth', 3, 15),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'model__min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }
    
    pipeline_lgbm = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', LGBMClassifier(
            random_state=RANDOM_STATE, 
            verbose=-1,
            class_weight='balanced'
        ))
    ])
    
    pipeline_lgbm.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_lgbm.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_lgbm.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [103]:
RUN_NAME_OPTUNE_LGBM = 'LGBMClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_LGBM, experiment_id=experiment_id) as run:
    run_id_lgbm = run.info.run_id

🏃 View run LGBMClassifier_optuna_als at: http://127.0.0.1:5000/#/experiments/1/runs/cfac6259063340ed9a861194b05f4f20
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [104]:
STUDY_NAME_LGBM = "LGBMClassifier_optuna_als"

mlflc_lgbm = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_lgbm}}
)

In [105]:
study_lgbm = optuna.create_study(direction='maximize', 
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                 study_name=STUDY_NAME_LGBM,
                                 storage=STUDY_DB_NAME,
                                 load_if_exists=True)

study_lgbm.optimize(objective_lgbm, 
                    n_trials=10, 
                    callbacks=[mlflc_lgbm]
                   )

best_params_lgbm = study_lgbm.best_params
print(f"Number of finished trials: {len(study_lgbm.trials)}")
print(f"Best params LGBM: {best_params_lgbm}")

[I 2026-05-04 03:08:08,181] Using an existing study with name 'LGBMClassifier_optuna_als' instead of creating a new one.


Средний NDCG: 0.8682
Средний Precision: 0.7952
Средний Recall: 0.8375
Средний F1-Score: 0.8053
Средний NDCG: 0.8576
Средний Precision: 0.7874
Средний Recall: 0.8291
Средний F1-Score: 0.7971


[I 2026-05-04 03:08:50,448] Trial 54 finished with value: 0.8632716854138354 and parameters: {'n_estimators': 700, 'max_depth': 15, 'learning_rate': 0.06268188700235208, 'num_leaves': 52, 'min_child_samples': 68}. Best is trial 19 with value: 0.8642042129240123.


Средний NDCG: 0.8633
Средний Precision: 0.7902
Средний Recall: 0.8323
Средний F1-Score: 0.8001
🏃 View run 54 at: http://127.0.0.1:5000/#/experiments/1/runs/cb7a54b65bf043fbbfb685cfbd431319
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8683
Средний Precision: 0.7703
Средний Recall: 0.8499
Средний F1-Score: 0.7952
Средний NDCG: 0.8575
Средний Precision: 0.7592
Средний Recall: 0.8399
Средний F1-Score: 0.7847


[I 2026-05-04 03:09:06,540] Trial 55 finished with value: 0.8628702370821726 and parameters: {'n_estimators': 100, 'max_depth': 12, 'learning_rate': 0.1543942786874978, 'num_leaves': 41, 'min_child_samples': 50}. Best is trial 19 with value: 0.8642042129240123.


...

Средний NDCG: 0.8623
Средний Precision: 0.8045
Средний Recall: 0.8099
Средний F1-Score: 0.7975
🏃 View run 61 at: http://127.0.0.1:5000/#/experiments/1/runs/6d999dacb7ac47c2aa5b0b5e26563711
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8682
Средний Precision: 0.7531
Средний Recall: 0.8537
Средний F1-Score: 0.7860
Средний NDCG: 0.8586
Средний Precision: 0.7458
Средний Recall: 0.8421
Средний F1-Score: 0.7770


[I 2026-05-04 03:14:31,712] Trial 62 finished with value: 0.86267727233082 and parameters: {'n_estimators': 400, 'max_depth': 8, 'learning_rate': 0.026688514879486674, 'num_leaves': 35, 'min_child_samples': 48}. Best is trial 19 with value: 0.8642042129240123.


Средний NDCG: 0.8627
Средний Precision: 0.7506
Средний Recall: 0.8457
Средний F1-Score: 0.7816
🏃 View run 62 at: http://127.0.0.1:5000/#/experiments/1/runs/3dcedb20d4a24022b49e7050a12cca20
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8684
Средний Precision: 0.7789
Средний Recall: 0.8493
Средний F1-Score: 0.8007
Средний NDCG: 0.8585
Средний Precision: 0.7662
Средний Recall: 0.8385
Средний F1-Score: 0.7888


[I 2026-05-04 03:15:18,820] Trial 63 finished with value: 0.8633488703319923 and parameters: {'n_estimators': 900, 'max_depth': 14, 'learning_rate': 0.02135368264395568, 'num_leaves': 46, 'min_child_samples': 40}. Best is trial 19 with value: 0.8642042129240123.


Средний NDCG: 0.8633
Средний Precision: 0.7747
Средний Recall: 0.8415
Средний F1-Score: 0.7948
🏃 View run 63 at: http://127.0.0.1:5000/#/experiments/1/runs/66e42cb49845477fbe2ae2ce00cdb3c5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 64
Best params LGBM: {'n_estimators': 900, 'max_depth': 13, 'learning_rate': 0.04019165012583699, 'num_leaves': 23, 'min_child_samples': 54}


In [106]:
pipeline_lgbm_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', LGBMClassifier(
        random_state=RANDOM_STATE, 
        verbose=-1,
        class_weight='balanced'
    ))
])

pipeline_lgbm_best.set_params(**study_lgbm.best_trial.user_attrs['pipeline_params'])
pipeline_lgbm_best.fit(X_train, y_train)

y_pred_proba_lgbm = pipeline_lgbm_best.predict_proba(X_test)

df_test_lgbm = df.loc[X_test.index].copy()
df_test_lgbm['y_pred_proba'] = y_pred_proba_lgbm[:, 1]

ndcg_lgbm_als, precision_lgbm_als, recall_lgbm_als, f1_lgbm_als = calculate_metrics(df_test_lgbm)
metrics_lgbm_als = {
    'NDCG': ndcg_lgbm_als,
    'Precision': precision_lgbm_als,
    'Recall': recall_lgbm_als,
    'F1': f1_lgbm_als
}

Средний NDCG: 0.7817
Средний Precision: 0.6875
Средний Recall: 0.7581
Средний F1-Score: 0.7078


In [107]:
RUN_NAME_LGBM = "best_optuna_lgbm_als"
REGISTRY_MODEL_NAME_LGBM = "best_optuna_lgbm_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_LGBM, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    lgbm_info = mlflow.sklearn.log_model(sk_model=pipeline_lgbm_best, 
                                        artifact_path='best_optuna_lgbm_als',
                                        registered_model_name=REGISTRY_MODEL_NAME_LGBM,
                                        input_example=input_example,
                                        code_paths=code_paths,
                                        await_registration_for=60
                                       )
    mlflow.log_metrics(metrics_lgbm_als)
    mlflow.log_params(best_params_lgbm)

Registered model 'best_optuna_lgbm_als' already exists. Creating a new version of this model...
2026/05/04 03:15:31 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_lgbm_als, version 6


🏃 View run best_optuna_lgbm_als at: http://127.0.0.1:5000/#/experiments/1/runs/48864bcdf75a48fe835206e5858cf7f4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '6' of model 'best_optuna_lgbm_als'.


## CatBoostClassifier

In [108]:
def objective_catboost(trial: optuna.Trial) -> float:
    params = {
        'model__iterations': trial.suggest_int('iterations', 100, 1000, step=50),
        'model__depth': trial.suggest_int('depth', 3, 10),
        'model__learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'model__l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True)
    }
    
    pipeline_catboost = Pipeline([
        ('preprocessing', ColumnTransformer([
            ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
        ], remainder='passthrough')),
        ('model', CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=0, 
            auto_class_weights='Balanced'
        ))
    ])
    
    pipeline_catboost.set_params(**params)
    
    kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    
    for train_idx, val_idx in kfold.split(X_train, y_train):
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        pipeline_catboost.fit(X_fold_train, y_fold_train)
        y_pred_proba = pipeline_catboost.predict_proba(X_fold_val)
        
        df_val = df.loc[X_fold_val.index].copy()
        df_val['y_pred_proba'] = y_pred_proba[:, 1]
        
        ndcg, _, _, _ = calculate_metrics(df_val)
        
        trial.set_user_attr('pipeline_params', params)
    
    return ndcg

In [109]:
RUN_NAME_OPTUNE_CATBOOST = 'CatBoostClassifier_optuna_als'

with mlflow.start_run(run_name=RUN_NAME_OPTUNE_CATBOOST, experiment_id=experiment_id) as run:
    run_id_catboost = run.info.run_id

🏃 View run CatBoostClassifier_optuna_als at: http://127.0.0.1:5000/#/experiments/1/runs/86c512c4ea5c4d62aa30f3bfa156e460
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [110]:
STUDY_NAME_CATBOOST = "CatBoostClassifier_optuna_als"

mlflc_catboost = MLflowCallback(
    tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
    metric_name="NDCG",
    create_experiment=False,
    mlflow_kwargs={'experiment_id': experiment_id, 'tags': {MLFLOW_PARENT_RUN_ID: run_id_catboost}}
)

In [111]:
study_catboost = optuna.create_study(direction='maximize', 
                                     sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                     study_name=STUDY_NAME_CATBOOST,
                                     storage=STUDY_DB_NAME,
                                     load_if_exists=True)

study_catboost.optimize(objective_catboost, 
                        n_trials=10, 
                        callbacks=[mlflc_catboost]
                       )

best_params_catboost = study_catboost.best_params
print(f"Number of finished trials: {len(study_catboost.trials)}")
print(f"Best params CatBoost: {best_params_catboost}")

[I 2026-05-04 03:15:31,545] Using an existing study with name 'CatBoostClassifier_optuna_als' instead of creating a new one.


Средний NDCG: 0.8679
Средний Precision: 0.7713
Средний Recall: 0.8502
Средний F1-Score: 0.7963
Средний NDCG: 0.8578
Средний Precision: 0.7596
Средний Recall: 0.8385
Средний F1-Score: 0.7844


[I 2026-05-04 03:15:54,666] Trial 50 finished with value: 0.8639132079826695 and parameters: {'iterations': 500, 'depth': 9, 'learning_rate': 0.04111995101217277, 'l2_leaf_reg': 3.0367083518552893}. Best is trial 47 with value: 0.8641061468851146.


Средний NDCG: 0.8639
Средний Precision: 0.7681
Средний Recall: 0.8444
Средний F1-Score: 0.7922
🏃 View run 50 at: http://127.0.0.1:5000/#/experiments/1/runs/af9ac230ab754beaa14963bff22176d5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8684
Средний Precision: 0.7702
Средний Recall: 0.8501
Средний F1-Score: 0.7960
Средний NDCG: 0.8578
Средний Precision: 0.7604
Средний Recall: 0.8395
Средний F1-Score: 0.7854


[I 2026-05-04 03:16:17,924] Trial 51 finished with value: 0.8637658992120882 and parameters: {'iterations': 500, 'depth': 9, 'learning_rate': 0.04112915945094078, 'l2_leaf_reg': 2.0913578983116703}. Best is trial 47 with value: 0.8641061468851146.


...

Средний NDCG: 0.8634
Средний Precision: 0.7753
Средний Recall: 0.8415
Средний F1-Score: 0.7952
🏃 View run 57 at: http://127.0.0.1:5000/#/experiments/1/runs/e01cbe3e7375489d8635ac3a0474431a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8680
Средний Precision: 0.7691
Средний Recall: 0.8492
Средний F1-Score: 0.7947
Средний NDCG: 0.8581
Средний Precision: 0.7569
Средний Recall: 0.8396
Средний F1-Score: 0.7833


[I 2026-05-04 03:19:27,504] Trial 58 finished with value: 0.8634134516551613 and parameters: {'iterations': 550, 'depth': 10, 'learning_rate': 0.02374255962099862, 'l2_leaf_reg': 1.2941246721556594}. Best is trial 47 with value: 0.8641061468851146.


Средний NDCG: 0.8634
Средний Precision: 0.7675
Средний Recall: 0.8436
Средний F1-Score: 0.7911
🏃 View run 58 at: http://127.0.0.1:5000/#/experiments/1/runs/630e287886b34270b74d9e09375fd97f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Средний NDCG: 0.8679
Средний Precision: 0.7641
Средний Recall: 0.8511
Средний F1-Score: 0.7923
Средний NDCG: 0.8577
Средний Precision: 0.7527
Средний Recall: 0.8406
Средний F1-Score: 0.7812


[I 2026-05-04 03:19:51,943] Trial 59 finished with value: 0.8640458162705477 and parameters: {'iterations': 700, 'depth': 8, 'learning_rate': 0.030974490355373658, 'l2_leaf_reg': 1.6061806994639913}. Best is trial 47 with value: 0.8641061468851146.


Средний NDCG: 0.8640
Средний Precision: 0.7656
Средний Recall: 0.8450
Средний F1-Score: 0.7909
🏃 View run 59 at: http://127.0.0.1:5000/#/experiments/1/runs/8fa680a9c75a46fba10afde3c7f68b64
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Number of finished trials: 60
Best params CatBoost: {'iterations': 400, 'depth': 9, 'learning_rate': 0.03471969203815024, 'l2_leaf_reg': 1.6639736464928652}


In [112]:
pipeline_catboost_best = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('categorical_encoding', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_features)
    ], remainder='passthrough')),
    ('model', CatBoostClassifier(
        random_state=RANDOM_STATE, 
        verbose=0, 
        auto_class_weights='Balanced'
    ))
])

pipeline_catboost_best.set_params(**study_catboost.best_trial.user_attrs['pipeline_params'])
pipeline_catboost_best.fit(X_train, y_train)

y_pred_proba_catboost = pipeline_catboost_best.predict_proba(X_test)

df_test_catboost = df.loc[X_test.index].copy()
df_test_catboost['y_pred_proba'] = y_pred_proba_catboost[:, 1]

ndcg_catboost_als, precision_catboost_als, recall_catboost_als, f1_catboost_als = calculate_metrics(df_test_catboost)
metrics_catboost_als = {
    'NDCG': ndcg_catboost_als,
    'Precision': precision_catboost_als,
    'Recall': recall_catboost_als,
    'F1': f1_catboost_als
}

Средний NDCG: 0.7809
Средний Precision: 0.6809
Средний Recall: 0.7593
Средний F1-Score: 0.7035


In [113]:
RUN_NAME_CATBOOST = "best_optuna_catboost_als"
REGISTRY_MODEL_NAME_CATBOOST = "best_optuna_catboost_als"

signature = mlflow.models.infer_signature(X_test, y_test)
input_example = X_test[:10]
code_paths = ["ML_Experiments.ipynb"]

with mlflow.start_run(run_name=RUN_NAME_CATBOOST, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    
    catboost_info = mlflow.sklearn.log_model(sk_model=pipeline_catboost_best, 
                                             artifact_path='best_optuna_catboost_als',
                                             registered_model_name=REGISTRY_MODEL_NAME_CATBOOST,
                                             input_example=input_example,
                                             code_paths=code_paths,
                                             await_registration_for=60
                                            )
    mlflow.log_metrics(metrics_catboost_als)
    mlflow.log_params(best_params_catboost)

Registered model 'best_optuna_catboost_als' already exists. Creating a new version of this model...
2026/05/04 03:20:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: best_optuna_catboost_als, version 6


🏃 View run best_optuna_catboost_als at: http://127.0.0.1:5000/#/experiments/1/runs/deb9ed32e2374720ae18649e5f87e79f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


Created version '6' of model 'best_optuna_catboost_als'.


## Model comparison

In [114]:
models_comparison = pd.DataFrame({
    'Model': ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM', 'CatBoost', 'XGBoost + ALS', 'LightGBM + ALS', 'CatBoost + ALS'],
    'NDCG': [ndcg_lr, ndcg_rf, ndcg_xgb, ndcg_lgbm, ndcg_catboost, ndcg_xgb_als, ndcg_lgbm_als, ndcg_catboost_als],
    'Precision': [precision_lr, precision_rf, precision_xgb, precision_lgbm, precision_catboost, precision_xgb_als, precision_lgbm_als, precision_catboost_als],
    'Recall': [recall_lr, recall_rf, recall_xgb, recall_lgbm, recall_catboost, recall_xgb_als, recall_lgbm_als, recall_catboost_als],
    'F1': [f1_lr, f1_rf, f1_xgb, f1_lgbm, f1_catboost, f1_xgb_als, f1_lgbm_als, f1_catboost_als]
})

models_comparison

,Model,NDCG,Precision,Recall,F1
0,LogisticRegression,0.751483,0.638633,0.598481,0.602124
1,RandomForest,0.778008,0.692051,0.726508,0.695558
2,XGBoost,0.779192,0.696847,0.741136,0.705449
3,LightGBM,0.780483,0.684553,0.748602,0.700788
4,CatBoost,0.779769,0.676177,0.751765,0.696813
5,XGBoost + ALS,0.781168,0.708642,0.742456,0.713721
6,LightGBM + ALS,0.781721,0.687478,0.758067,0.707797
7,CatBoost + ALS,0.780935,0.680914,0.759255,0.703533


In [115]:
best_model_idx = models_comparison['NDCG'].idxmax()
print(f"\nЛучшая модель: {models_comparison.iloc[best_model_idx]['Model']} с NDCG = {models_comparison.iloc[best_model_idx]['NDCG']:.4f}")


Лучшая модель: LightGBM + ALS с NDCG = 0.7817


<div style="background-color: #98FB98; color: black; padding: 10px; border-radius: 5px;">

Сохраним обученный итоговый пайплайн.

</div>

In [116]:
MODEL_NAME = 'pipeline_lgb_best.pkl'
with open(MODEL_NAME, 'wb') as file:
    pickle.dump(pipeline_lgbm_best, file)

In [130]:
def show_vacancy_predictions(pipeline, X_test, y_test, df,
                             n_top=10, vacancy_id=None, random_state=None):
    """
    Показывает предсказания пайплайна для выбранной вакансии из тест-сета.

    Args:
        pipeline     : обученный sklearn Pipeline
        X_test       : тестовые признаки (те же колонки, на которых обучался pipeline)
        y_test       : истинные метки
        df           : исходный датафрейм со всеми колонками (для отображения)
        n_top        : сколько топ-резюме показать
        vacancy_id   : None = случайная вакансия; иначе конкретный ID
        random_state : seed для воспроизводимости случайного выбора

    Returns:
        pd.DataFrame, отсортированный по pred_proba убыванием
    """
    df_test = df.loc[X_test.index].copy()
    df_test['target'] = y_test.values

    if vacancy_id is None:
        rng = np.random.RandomState(random_state)
        vacancy_id = rng.choice(df_test['vacancy_id'].unique())

    mask = df_test['vacancy_id'] == vacancy_id
    if not mask.any():
        print(f"[!] vacancy_id={vacancy_id} не найден в тестовой выборке.")
        return None

    df_vac = df_test[mask].copy()
    X_vac  = X_test.loc[df_vac.index]

    df_vac['pred_proba'] = pipeline.predict_proba(X_vac)[:, 1]
    df_vac = df_vac.sort_values('pred_proba', ascending=False).reset_index(drop=True)

    r0       = df_vac.iloc[0]
    vac_name = str(r0.get('vacancy_name', '—'))
    vac_desc = str(r0.get('vacancy_description', '—'))
    SEP, SEP2 = "=" * 90, "─" * 90

    print(SEP)
    print(f"  ВАКАНСИЯ : {vac_name}   [id={vacancy_id}]")
    print(f"  Опыт     : {r0.get('vacancy_experience','—')}  |  "
          f"Занятость : {r0.get('vacancy_employment','—')}  |  "
          f"График : {r0.get('vacancy_schedule','—')}")
    print(SEP2)
    print("  ОПИСАНИЕ ВАКАНСИИ:")
    for line in vac_desc[:1200].split('\n')[:25]:
        if line.strip():
            print(f"    {line.strip()}")
    if len(vac_desc) > 1200:
        print("    [... сокращено]")
    print(SEP)

    n_pos = int(df_vac['target'].sum())
    print(f"\n  ТОП-{n_top} из {len(df_vac)} кандидатов  "
          f"(всего реально подходящих в выборке: {n_pos})\n")

    for rank, (_, row) in enumerate(df_vac.head(n_top).iterrows(), 1):
        icon   = "✅" if row['target'] == 1 else "❌"
        exp    = str(row.get('resume_last_experience_description', '—'))
        skills = str(row.get('resume_skills', '—'))
        print(SEP2)
        print(f"  #{rank}  {icon}  score={row['pred_proba']:.4f}  "
              f"target={int(row['target'])}  [resume_id={row.get('resume_id','—')}]")
        print(f"  Должность : {row.get('resume_last_position','—')}")
        print(f"  Локация   : {row.get('resume_location','—')}  |  "
              f"Опыт: {row.get('resume_experience_months','—')} мес.")
        print(f"  Навыки    : {skills[:180]}{'...' if len(skills) > 180 else ''}")
        print(f"  Описание последнего места:")
        for line in exp[:700].split('\n')[:14]:
            if line.strip():
                print(f"    {line.strip()}")
        if len(exp) > 700:
            print("    [... сокращено]")
        print()

    print(SEP2)
    n_hit = int(df_vac.head(n_top)['target'].sum())
    print(f"\n  Попаданий в топ-{n_top}: {n_hit}/{n_top} ({100*n_hit//n_top}%)")
    print(f"  Всего релевантных в тест-сете для этой вакансии: {n_pos}\n")

    return df_vac

In [138]:
# Случайная вакансия из тест-сета, лучший пайплайн
result = show_vacancy_predictions(pipeline_lgbm_best, X_test, y_test, df,
                                  n_top=10, vacancy_id=126372900)

# Зафиксировать вакансию и сравнить другой пайплайн:
# vacancy_id_fixed = result.iloc[0]['vacancy_id']
# show_vacancy_predictions(pipeline_lgbm_als, X_test, y_test, df,
#                          n_top=10, vacancy_id=vacancy_id_fixed)

  ВАКАНСИЯ : Интернет-Маркетолог в Wildcrm   [id=126372900]
  Опыт     : От 3 до 6 лет  |  Занятость : Полная занятость  |  График : Удаленная работа
──────────────────────────────────────────────────────────────────────────────────────────
  ОПИСАНИЕ ВАКАНСИИ:
    О компании WildCRM — новый B2B-продукт для оцифровки финансовых данных маркетплейсов. Мы работаем с крупнейшими представителями рынка, такими как Wildberries и Ozon. Упрощаем финансовую отчетность и даем бизнесу возможность принимать data-driven решения. Мы хотим вырастить тебя с миддла до лида, который сможет возглавить направление интернет-маркетинга!   С нас:  Доход 130к- 150к Полная удаленка Рост от эксперта до лида команды Минимум бюрократии Поддержка коллег-маркетологов Команда исполнителей Атмосфера стартапа, стабильность зрелой компании    С тебя:  Запускать и масштабировать Telegram Ads Настраивать сквозную аналитику Быть &quot;руками&quot; CEO в создании и развитии продукта, управлять подрядчиками - командами разра